In [1]:
from typing import List, Dict
import ipywidgets as widgets
import os
import os.path as osp
from collections import OrderedDict

In [2]:
data_dirs = [
    "/data/mid",
    "/data/processed"
]

In [3]:
class DataSelector:
    def __init__(self, data_dirs:List[str], desciption:str=None,
                 type_field=False):
        """
        data_dirs (list[str]): list of dirs with .csv files
        """
        print("Read data dirs...", flush=True)
        datas = OrderedDict()
        for data_dir in data_dirs:
            files = list(filter(lambda x: x.endswith(".csv"), os.listdir(data_dir)))
            files = sorted(files)
            print(f"{data_dir}: {len(files)} files")
            # ["name.v{X}.csv"]
            names = dict()
            for file in files:
                sep = file.index(".")
                name, vers = file[ :sep], file[sep: ]
                try:
                    names[name].append(vers)
                except KeyError:
                    names[name] = [vers]
            if names:
                datas[data_dir] = names
        self.datas = datas
        self.data_dirs = list(datas.keys())
        self.type_field = type_field
        self._init_widgets(desciption)

    def _init_widgets(self, desciption:str=None):
        layout = {'width': 'max-content'}
        self.title = widgets.Label(desciption) \
                     if desciption else None
        self.dir_select = widgets.Dropdown(options=self.data_dirs,
                                      value=None,
                                      disabled=False,
                                      layout=layout)
        self.name_select = widgets.Dropdown(disabled=True, layout=layout)
        self.vers_select = widgets.Dropdown(disabled=True, layout=layout)
        if self.type_field:
            place_hold = '/path/to/data.csv   regex are available'
            self.inp_field = widgets.Text(value=None, disabled=False,
                                          placeholder=place_hold)
        else:
            self.inp_field = widgets.Label("")
        # event trigger
        self.dir_select.observe(self._dir_change, names='value')
        self.name_select.observe(self._name_change, names='value')
        self.vers_select.observe(self._vers_change, names='value')

    def _update_path(self):
        dir_, name = self.dir_select.value, self.name_select.value
        vers = self.vers_select.value
        input_path = osp.join(dir_, name) + vers
        assert osp.exists(input_path), f"'{input_path}'"
        self.inp_field.value = input_path
        
    def _dir_change(self, event):
        data_dir = event.new
        names = list(self.datas[data_dir].keys())
        self.name_select.options = names
        self.name_select.disabled = False

    def _name_change(self, event):
        name = event.new
        data_dir = self.dir_select.value
        vers = list(self.datas[data_dir][name])
        self.vers_select.options = vers
        if len(vers) == 1:
            self.vers_select.value = vers[0]
            self.vers_select.disabled = True
            self._update_path()
        else:
            self.vers_select.options = vers
            self.vers_select.disabled = False

    def _vers_change(self, event):
        self._update_path()

    def display(self):
        hbox = widgets.HBox([self.dir_select, 
                             self.name_select, 
                             self.vers_select])
        vbox = [hbox, self.inp_field]
        if self.title:
            vbox.insert(0, self.title)
        vbox = widgets.VBox(vbox)
        return vbox

    def get_widget(self):
        return self.display()

    def get_state(self) -> str:
        return self.inp_field.value

In [4]:
class AddParams:
    def __init__(self, params:Dict[str, type]):
        wdgs = []
        self.params = dict()
        for name, t in params.items():
            if t == bool:
                wdg = widgets.Checkbox(
                    value=False,
                    description=name,
                    disabled=False,
                    indent=False
                )
            elif t == int:
                wdg = widgets.IntText(
                        description=name,
                        disabled=False,
                        layout={'width': "20ex"},
                    )
            elif t == str:
                wdg = widgets.Text(
                        disabled=False,
                        layout={'width': "28ex"},
                        description=name
            ) 
            elif isinstance(t, list):
                wdg = widgets.Dropdown(options=t,
                                       disabled=False,
                                       description=name, 
                                       layout={'width': 'max-content'})
            else:
                raise ValueError(f"Invalid param type '{t}'")
            wdgs.append(wdg)
            self.params[name] = wdg

        self.wdgs = wdgs

    def display(self):
        return widgets.VBox(self.wdgs)

    def get_widget(self):
        return self.display()

## Run train

In [5]:
train_sh = "/mnt/nvme/vovik/noise_classification/docker/train.sh"
# docker run
train_command = "docker run -d --name {container_name} "\
                "--shm-size=32gb" \
                '--network="host" -v /mnt:/mnt '\
                "-v $PWD/../:/app/ "\
                "-v /mnt/raid10/datasets/projects/noise_classification:/data "\
                "-w /app/ "\
                "--user $(id -u):$(id -g) "\
                '--gpus "device=8" '\
                "audio-image:clap "\
                "python3 -m noisecls.train "
train_params = {
    "container_name": str,
    "batch_size": int,
    "epochs": int,
    "experiment": str,
    "comment": str,
    "clearml": bool,
    
}

In [6]:
def get_next_version(fpath:str) -> str:
    """
    train_{X}.sh
    """
    dir_ = osp.split(fpath)[0]
    vers = 1
    while osp.exists(fpath):
        fpath = osp.join(dir_, f"train_{vers}.sh")
        vers += 1
    return fpath

def docker_run(
        train_data:str,
        test_data:str, 
        container_name:str="train-noises",
        epochs:int=15,
        batch_size:int=20,
        experiment:str=None,
        comment:str=None,
        clearml:bool=False):
    command = train_command.format(container_name=container_name) + \
                f"--train-data {train_data} " \
                f"--test-data {test_data} " \
                f"--batch-size {batch_size} " \
                f"--epochs {epochs}"
    # additional
    if experiment:
        command += f" -exp {experiment}"
    if comment:
        command += f" -m {comment}"
    if clearml:
        command += f" --clearml"

    # or use os.system on host (not in container)
    sh_file = get_next_version(train_sh)
    with open(sh_file, 'w') as f:
        f.write("#!/bin/bash" + "\n\n")
        f.write(command)
    print("Train command is written to", sh_file)

In [7]:
train_data_selector = DataSelector(data_dirs, type_field=False, desciption="Train data")
test_data_selector = DataSelector(data_dirs, type_field=False, desciption="Test data")

params_selector = AddParams(train_params)

Read data dirs...
/data/mid: 26 files
/data/processed: 13 files
Read data dirs...
/data/mid: 26 files
/data/processed: 13 files


In [8]:
# Final button
button = widgets.Button(description="Train", 
                        button_style='success' # 'success', 'info', 'warning', 'danger', ''
                ) 
def run(b):
    args = dict()
    for param, wdg in params_selector.params.items():
        args[param] = wdg.value
    args["train_data"] = train_data_selector.get_state()
    args["test_data"] = test_data_selector.get_state()
    docker_run(**args)

button.on_click(run) # Назначаем этот обработчик на событие "on_click"

widgets.VBox([
    train_data_selector.get_widget(), 
    test_data_selector.get_widget(), 
    params_selector.get_widget(),
    button])